# RosenblattVinecop demo

This notebook demonstrates conditional vine-copula estimation with `RosenblattVinecop`. We simulate observations from a three-dimensional Gaussian copula whose equicorrelation changes with a scalar covariate $x$, fit every registered non-finetuning backend to the same data and fixed R-vine structure, and compare their fitting times, conditional densities, and samples.

The pointwise API uses an `u` matrix with one column per copula margin. The optional conditioning matrix `x` has one row per observation and must use the same NumPy or torch array namespace as `u` within a call.

## Setup

Install one PyTorch flavour together with the optional backends, for example `uv sync --extra cu130 --extra backends`. TabPFN, TabICL, and Nori may require credentials, model downloads, substantial memory, or a GPU. A three-dimensional vine fits three pair copulas, each of which trains both conditional directions, so this comparison is more expensive than the pair-copula demo.

Every registered non-finetuning backend is attempted below. An unavailable backend or failed fit is recorded and skipped so the remaining comparisons can still run.

In [ ]:
from time import perf_counter

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import pyvinecopulib as pv
import torch
from dotenv import load_dotenv
from scipy.stats import norm

from npcc import (
  QuantileGridConfig,
  RosenblattVinecop,
  available_backends,
)

load_dotenv()

SEED = 42
N_TRAIN = 400
N_TEST = 8
RHO_MIN, RHO_MAX = 0.10, 0.80
X_MIN, X_MAX = 0.05, 0.95
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

rng = np.random.default_rng(SEED)
print(f"device: {DEVICE}")

## Conditional Gaussian-copula data

For each row, all three latent normal variables have correlation

$$\rho(x)=0.1+0.7x.$$

The factor representation $Z_j=\sqrt{\rho(x)}Z_0+\sqrt{1-\rho(x)}\epsilon_j$ produces a positive-definite equicorrelation matrix for every $x$ in the selected interval. Applying the standard-normal CDF to each $Z_j$ gives uniform copula margins.

In [ ]:
def rho_of_x(x: np.ndarray) -> np.ndarray:
  x = np.asarray(x, dtype=np.float64)
  return RHO_MIN + (RHO_MAX - RHO_MIN) * x


def sample_conditional_gaussian_copula(
  x: np.ndarray, rng: np.random.Generator
) -> np.ndarray:
  x = np.asarray(x, dtype=np.float64).reshape(-1)
  rho = rho_of_x(x)[:, None]
  common = rng.standard_normal((x.size, 1))
  noise = rng.standard_normal((x.size, 3))
  z = np.sqrt(rho) * common + np.sqrt(1.0 - rho) * noise
  return np.clip(norm.cdf(z), 1e-9, 1.0 - 1e-9)


def gaussian_copula_pdf(u: np.ndarray, x: np.ndarray) -> np.ndarray:
  u = np.asarray(u, dtype=np.float64)
  x = np.asarray(x, dtype=np.float64).reshape(-1)
  z = norm.ppf(np.clip(u, 1e-12, 1.0 - 1e-12))
  rho = rho_of_x(x)
  correlation = np.full((x.size, 3, 3), 1.0, dtype=np.float64)
  correlation *= rho[:, None, None]
  diagonal = np.arange(3)
  correlation[:, diagonal, diagonal] = 1.0
  inverse_difference = np.linalg.inv(correlation) - np.eye(3)
  quadratic = np.einsum("ni,nij,nj->n", z, inverse_difference, z)
  determinant = np.linalg.det(correlation)
  return np.exp(-0.5 * quadratic) / np.sqrt(determinant)


x_train = rng.uniform(X_MIN, X_MAX, size=(N_TRAIN, 1))
u_train = sample_conditional_gaussian_copula(x_train, rng)
x_test = np.linspace(0.10, 0.90, N_TEST)[:, None]
u_test = sample_conditional_gaussian_copula(x_test, rng)

fig, axes = plt.subplots(1, 3, figsize=(14, 4), sharex=True, sharey=True)
for ax, (left, right) in zip(axes, [(0, 1), (0, 2), (1, 2)]):
  points = ax.scatter(
    u_train[:, left],
    u_train[:, right],
    c=x_train[:, 0],
    s=12,
    alpha=0.65,
  )
  ax.set(
    xlabel=f"u{left + 1}",
    ylabel=f"u{right + 1}",
    title=f"Margins {left + 1} and {right + 1}",
    xlim=(0, 1),
    ylim=(0, 1),
  )
  ax.set_aspect("equal")
fig.colorbar(points, ax=axes, label="x", shrink=0.85)
fig.suptitle("Conditional Gaussian-copula training data")
plt.show()

## Fixed R-vine structure

`RosenblattVinecop` fits the pair copulas along a supplied structure; it does not select the structure automatically. Here every backend uses the same full three-dimensional D-vine with order $(1,2,3)$.

In [ ]:
structure = pv.RVineStructure.from_order([1, 2, 3])

print(f"dimension: {structure.dim}")
print(f"truncation level: {structure.trunc_lvl}")
print(f"order: {structure.order}")
print("structure matrix:")
print(structure.matrix)

## Fit all non-finetuning backends

The backend list comes from the registry rather than a handwritten subset. Names ending in `-finetune` are excluded. Lightweight settings keep this demonstration tractable, while a common quantile grid and target transform make the fits as comparable as their different model families allow.

GPU operations are asynchronous, so CUDA is synchronized on both sides of each timer. Failed attempts retain their elapsed time and error in the report.

In [ ]:
BACKEND_KWARGS = {
  "catboost": {"iterations": 100, "random_seed": SEED},
  "gbm": {"n_estimators": 100, "max_depth": 3, "random_state": SEED},
  "ngboost": {
    "n_estimators": 300,
    "learning_rate": 0.03,
    "random_state": SEED,
  },
  "pytabkit-realmlp": {"n_epochs": 8, "random_state": SEED},
  "pytabkit-tabm": {"n_epochs": 8, "random_state": SEED},
  "tabicl": {"model_kwargs": {"n_estimators": 1}},
}
BACKENDS = [
  name for name in available_backends() if not name.endswith("-finetune")
]
QUANTILE_CONFIG = QuantileGridConfig(n_quantiles=41)


def synchronize_device() -> None:
  if DEVICE.type == "cuda":
    torch.cuda.synchronize(DEVICE)


models: dict[str, RosenblattVinecop] = {}
fit_records: list[dict[str, object]] = []
for backend in BACKENDS:
  synchronize_device()
  started = perf_counter()
  try:
    model = RosenblattVinecop.from_data(
      u_train,
      structure,
      x=x_train,
      backend=backend,
      quantile_config=QUANTILE_CONFIG,
      transform="logit",
      device=DEVICE,
      backend_kwargs=BACKEND_KWARGS.get(backend),
    )
    synchronize_device()
    elapsed = perf_counter() - started
    models[backend] = model
    fit_records.append(
      {
        "backend": backend,
        "status": "ok",
        "fit_seconds": elapsed,
        "error": "",
      }
    )
  except Exception as exc:
    synchronize_device()
    elapsed = perf_counter() - started
    fit_records.append(
      {
        "backend": backend,
        "status": "skipped",
        "fit_seconds": elapsed,
        "error": f"{type(exc).__name__}: {exc}",
      }
    )

fit_report = pd.DataFrame(fit_records)
display(fit_report.style.format({"fit_seconds": "{:.2f}"}))

if not models:
  raise RuntimeError("No backend could be fitted; inspect the report above.")

## Conditional PDF on a shared test set

All successful models evaluate the same eight observations and covariates. Because the data-generating copula is known, its analytic density provides a reference. We report mean absolute error (MAE) on the density scale and mean absolute log-density error; the latter clips values only to make the logarithm finite. With eight test rows, these numbers illustrate the API and should not be interpreted as a benchmark.

In [ ]:
truth_pdf = gaussian_copula_pdf(u_test, x_test)
estimated_pdf = {
  backend: np.asarray(model.pdf(u_test, x=x_test))
  for backend, model in models.items()
}

pdf_report = pd.DataFrame(
  {
    "test_row": np.arange(N_TEST),
    "x": x_test[:, 0],
    "u1": u_test[:, 0],
    "u2": u_test[:, 1],
    "u3": u_test[:, 2],
    "truth": truth_pdf,
    **estimated_pdf,
  }
)
display(pdf_report.round(4))

LOG_EPS = 1e-12
pdf_metrics = pd.DataFrame(
  [
    {
      "backend": backend,
      "mae": np.mean(np.abs(values - truth_pdf)),
      "mean_absolute_log_error": np.mean(
        np.abs(
          np.log(np.clip(values, LOG_EPS, None))
          - np.log(np.clip(truth_pdf, LOG_EPS, None))
        )
      ),
    }
    for backend, values in estimated_pdf.items()
  ]
).sort_values("mae")
display(pdf_metrics.round(4))

fig, ax = plt.subplots(figsize=(9, 5))
ax.plot(
  pdf_report["test_row"],
  truth_pdf,
  "ko--",
  lw=2.5,
  label="truth",
)
for backend, values in estimated_pdf.items():
  ax.plot(pdf_report["test_row"], values, marker="o", label=backend)
ax.set(
  xlabel="test row",
  ylabel="conditional copula density",
  title="PDF estimates on the shared test set",
)
ax.grid(alpha=0.25)
ax.legend(bbox_to_anchor=(1.02, 1), loc="upper left")
fig.tight_layout()
plt.show()

## Conditional samples

`sample(n, x=...)` associates one generated copula row with each row of `x`, so here `n` equals the number of test covariates. Sampling is torch-native: the covariates are therefore moved to the vine's device before the call. The same seed is used for each backend, making this example reproducible and giving the models the same base random numbers.

In [ ]:
x_test_tensor = torch.as_tensor(x_test, dtype=torch.float64, device=DEVICE)
samples_by_backend = {
  backend: model.sample(N_TEST, x=x_test_tensor, seeds=[SEED])
  .detach()
  .cpu()
  .numpy()
  for backend, model in models.items()
}

sample_report = pd.concat(
  [
    pd.DataFrame(
      {
        "backend": backend,
        "test_row": np.arange(N_TEST),
        "x": x_test[:, 0],
        "u1": samples[:, 0],
        "u2": samples[:, 1],
        "u3": samples[:, 2],
      }
    )
    for backend, samples in samples_by_backend.items()
  ],
  ignore_index=True,
)
display(sample_report.round(4))

sample_checks = pd.DataFrame(
  [
    {
      "backend": backend,
      "shape": str(samples.shape),
      "all_finite": bool(np.all(np.isfinite(samples))),
      "inside_unit_cube": bool(np.all((samples >= 0.0) & (samples <= 1.0))),
    }
    for backend, samples in samples_by_backend.items()
  ]
)
display(sample_checks)

## Takeaways

- `RosenblattVinecop.from_data` fits a common fixed structure with any registered conditional-distribution backend.
- NumPy `u` and `x` inputs produce NumPy PDF values, while conditional sampling uses torch tensors on the configured device.
- The fitting report keeps optional dependency, credential, download, and runtime failures visible without discarding successful models.
- The tiny test set is intended to show the interface. Use a larger held-out set and repeated conditional samples for model comparison or calibration studies.